In [ ]:
import os, glob

import cytoflow as flow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from flow_analysis_tools import flow_violin
from flow_analysis_tools.color_palettes import *

from rclone_tools import NodeExperimentsDirectory

In [ ]:
DRIVE_RELPATH = "ARS_Working/Experiments/6_phHSC_Testing/260914_phHSCc12R1_Test/260914_phHSCc12R1_multi-hyper_Vp1d3/analysis"
DATA_DIR = "/data_gilbert3/asathler/hsiung-box"
PROXY = "http://169.230.134.132:3128"

In [ ]:
Experiment = NodeExperimentsDirectory(
    drive="hsiung",
    source_relpath=DRIVE_RELPATH,
    node_mountpath=DATA_DIR,
    output_dirname="out",
    proxy=PROXY
)

In [ ]:
Experiment.stage()

In [ ]:
%matplotlib widget
matplotlib.rc('figure', dpi = 160)
plt.style.use('ggplot')

In [ ]:
FCS_FILES = sorted(glob.glob(os.path.join(Experiment.dest_dirpath, "*.fcs")))
YLIMITS = (0, 7)
CHANNELS = {
    "VL1-A": ["Cas12", "mTagBFP2"],
    "BL1-A": ["Empty", "None"],
    "RL1-A": ["Empty", "None"],
    "YL1-A": ["Empty", "None"]
}
CHANNEL_CMAPS = {
    "VL1-A": black_blue_pastel_palette,
    "BL1-A": black_green_pastel_palette,
    "RL1-A": black_red_pastel_palette,
    "YL1-A": black_yellow_pastel_palette,
}
ALL_NEG_CONSTRUCT = "None"
GATE_INFO = {}
print('\n'.join([f"'{x}'" for x in FCS_FILES]))
if isinstance(Experiment, NodeExperimentsDirectory):
    for x in FCS_FILES: Experiment.logger.info(x)

In [ ]:
CONSTRUCT_N_UNDERSCORE = -2
SECONDARY_N_UNDERSCORE = -1

def get_construct_name(name):
    return name.split("_")[CONSTRUCT_N_UNDERSCORE]

def get_secondary_name(name):
    name = os.path.splitext(name)[0]
    return name.split("_")[SECONDARY_N_UNDERSCORE]

In [ ]:
FCS_NAMES = [os.path.basename(filename) for filename in FCS_FILES]
SAMPLE_NAMES = [get_construct_name(name) for name in FCS_NAMES]
SECONDARY_NAMES = [get_secondary_name(name) for name in FCS_NAMES]
max_fcs_len = max([len(x) for x in FCS_NAMES]) + 5
for fcs, sample_name, secondary_name in zip(FCS_NAMES, SAMPLE_NAMES, SECONDARY_NAMES):
    print_txt = f"{fcs + ''.join([' '] * (max_fcs_len - len(fcs)))}{str(sample_name):<15}{str(secondary_name):<15}"
    print(print_txt)
    if isinstance(Experiment, NodeExperimentsDirectory):
        Experiment.logger.info(print_txt)

In [ ]:
import_op = flow.ImportOp(
    conditions = {
        "Construct": "category",
        "Secondary": "category"
    },
    tubes = [
        flow.Tube(file=fcs_file, conditions={
            "Construct": get_construct_name(fcs_file),
            "Secondary": get_secondary_name(fcs_file)
    }) for fcs_file in FCS_FILES],
    # channels = channel_names
)

ex = import_op.apply()

# Gating Single Cells

In [ ]:
single_gate = flow.PolygonOp(
    name="Single_Cell",
    xchannel="FSC-A",
    ychannel="SSC-A"
)

single_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True
).plot(ex, gridsize = 100)

In [ ]:
if isinstance(Experiment, NodeExperimentsDirectory):
    plt.savefig(Experiment.output_dirpath / f"single_cell_gate.png")

In [ ]:
GATE_INFO["single_cell_vertices"] = np.asarray(single_gate.vertices)
ex = single_gate.apply(ex)
plt.close()

# Timeseries
Based on secondary

In [ ]:
%matplotlib inline

In [ ]:
plot_data = ex.data[ex.data["Single_Cell"] == True]
PLOT_CHANNEL = "VL1-A"
PALETTE_MAP = {
    "pARS001": black_blue_pastel_palette,
    "pCH45": black_green_pastel_palette,
    "pRA02": black_red_pastel_palette
}

In [ ]:
fig, axs = plt.subplot_mosaic(
    [
        ["Control", "S1", "S2", "S3", "Transduction"]
    ],
    figsize=(12,5),
    width_ratios=(2, 1, 1, 1, 9),
    sharey=True
    # layout="consrained"
)

flow_violin(
    data=plot_data.loc[(plot_data["Construct"] == "phHSC") | (plot_data["Construct"] == "phHSC2")],
    x="Construct",
    # y=flow_channel,
    y=PLOT_CHANNEL,
    # hue="Guide",
    color=black_blue_pastel_palette[0],
    log_y_axis=True,
    order=["phHSC", "phHSC2"],
    ylabel=PLOT_CHANNEL,
    xlabel="",
    density_norm='width',
    ylim=YLIMITS,
    ax=axs['Control']
)

for cdx, this_construct in enumerate(["pARS001", "pCH45", "pRA02"]):
    flow_violin(
        data=plot_data.loc[
            (plot_data["Secondary"] == "spinPoly") & 
            (plot_data["Construct"] == this_construct)
        ],
        x="Construct",
        # y=flow_channel,
        y=PLOT_CHANNEL,
        # hue="Construct",
        color=PALETTE_MAP[this_construct][1],
        log_y_axis=True,
        # ylabel=fluor,
        xlabel="",
        density_norm='width',
        ylim=YLIMITS,
        ax=axs[f"S{cdx+1}"]
    )

final_data = plot_data.loc[
    ~plot_data["Construct"].isin(["phHSC", "phHSC2"]) &
    ~plot_data["Secondary"].isin(["spinPoly", "None"]) & 
    plot_data["Secondary"].notna()
].copy()
if final_data["Construct"].dtype.name == "category":
    final_data["Construct"] = final_data["Construct"].cat.remove_unused_categories()
assert "phHSC" not in final_data["Construct"].unique()
final_data.loc[:, "Secondary"] = pd.to_numeric(final_data.loc[:, "Secondary"])
final_data.loc[:, "Secondary"] = round((final_data.loc[:, "Secondary"] / (1000 + final_data.loc[:, "Secondary"])) * 100) + 1
log_y = f"{PLOT_CHANNEL}_logy"
# print(final_data[PLOT_CHANNEL])
final_data.loc[final_data[PLOT_CHANNEL] == 0, PLOT_CHANNEL] = 1
final_data.loc[:, log_y] = np.log10(final_data.loc[:, PLOT_CHANNEL])
# final_data[log_y] = final_data[log_y].fillna(0)
# print(final_data[log_y])

summary_df = final_data.groupby(["Construct", "Secondary"]).mean().reset_index()

sns.stripplot(
    data=final_data,
    x="Secondary",
    y=log_y,
    native_scale=True,
    hue="Construct",
    palette=[cmap[1] for cmap in PALETTE_MAP.values()],
    size=2,
    alpha=0.5,
    # linewidth=0.1,
    # edgecolor="black"
    ax=axs["Transduction"],
    jitter=0.45,
    legend=False
)

# print(summary_df["Secondary"])
# print(summary_df[log_y])
for cdx, this_construct in enumerate(["pARS001", "pCH45", "pRA02"]):
    axs["Transduction"].scatter(
        summary_df.loc[summary_df["Construct"] == this_construct, "Secondary"],
        summary_df.loc[summary_df["Construct"] == this_construct, log_y],
        color=PALETTE_MAP[this_construct][1],
        zorder=5,
        # alpha=0.5,
        edgecolor="k",
        linewidths=1,
        label=this_construct
    )

# sns.lineplot(
#     # data=summary_df,
#     data=final_data,
#     x="Secondary",
#     hue="Construct",
#     y=log_y,
#     ax=axs["Transduction"],
#     palette=[cmap[1] for cmap in PALETTE_MAP.values()],
#     # size=12,
#     markers=["o", "o", "o"],
#     style="Construct"
#     # edgecolor=np.clip(np.array(black_red_pastel_palette[0]) - [0.25, 0.25, 0.25], a_min=0, a_max=1)
# )

axs["Transduction"].set_xlabel("Dosage (% 1x viral sup.)")
plt.legend(ncol=3, loc="lower right")
plt.savefig(Experiment.output_dirpath / f"plot.png")

In [ ]:
plot_data.to_csv(Experiment.output_dirpath / f"processed_data.csv")
plot_data.to_csv(Experiment.output_dirpath / f"groupby_means.csv")

In [ ]:
Experiment.close()